In [ ]:
# train_fft_detector_on_the_fly.py
# Requirements: torch, torchvision, PIL, tqdm, sklearn
# Run in same session where `pipe` and `text_embeddings` are defined.
import os
from pathlib import Path
import random
import io
import math

import diffusers
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import functional as TF
from PIL import Image, ImageFilter
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
# Suppress UserWarning
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------- CONFIG -----------------
DATA_DIR = "./watermark_dataset"  # existing dataset folder (do NOT recreate)
BATCH_SIZE = 8
NUM_WORKERS = 0  # set 0 in notebooks/windows (avoid pickling pipe)
NUM_EPOCHS = 50
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
VALIDATION_SPLIT = 0.15
IMAGE_SIZE = 512  # might adjust to your pipeline / VAE size
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5

import torch
from diffusers import DPMSolverMultistepScheduler
from inverse_stable_diffusion import InversableStableDiffusionPipeline

model_id = "stabilityai/stable-diffusion-2-1-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
scheduler = DPMSolverMultistepScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe = InversableStableDiffusionPipeline.from_pretrained(
    model_id,
    scheduler=scheduler,
    torch_dtype=torch.float16,
    revision="fp16",
    verbose=False,
)
diffusers.utils.logging.disable_progress_bar()
pipe.set_progress_bar_config(disable=True)
pipe = pipe.to(device)

TEXT_EMBEDDINGS = pipe.get_text_embedding("")  #
PIPE = pipe  # make sure 'pipe' is in scope
# ------------------------------------------

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if PIPE is None:
    raise RuntimeError(
        "Please ensure `pipe` variable is available in scope before running this script."
    )
if TEXT_EMBEDDINGS is None:
    raise RuntimeError(
        "Please ensure `text_embeddings` variable is available in scope before running this script."
    )


# ---------------- Augmentations (image-space) ----------------
# Enriched train_transforms — each optional action wrapped in RandomApply
def jpeg_compress_pil(img: Image.Image, quality: int = 85):
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    buf.seek(0)
    return Image.open(buf).convert("RGB")


class RandomJPEG:
    def __init__(self, p=0.5, q_range=(60, 95)):
        self.p = p
        self.q_range = q_range

    def __call__(self, img):
        if random.random() < self.p:
            q = random.randint(self.q_range[0], self.q_range[1])
            return jpeg_compress_pil(img, q)
        return img


class RandomGaussianNoise:
    def __init__(self, p=0.5, std=0.01):
        self.p = p
        self.std = std

    def __call__(self, img):
        if random.random() < self.p:
            arr = np.array(img).astype(np.float32) / 255.0
            noise = np.random.normal(0, self.std, arr.shape).astype(np.float32)
            arr = np.clip(arr + noise, 0.0, 1.0)
            img2 = Image.fromarray((arr * 255).astype(np.uint8))
            return img2
        return img


def make_train_image_augmentations(IMAGE_SIZE):
    # Compose PIL-based augmentations (randomly applied)
    aug_list = []
    # random rotation small
    aug_list.append(
        transforms.RandomApply([transforms.RandomRotation(degrees=15)], p=0.5)
    )
    # random resized crop (sometimes)
    aug_list.append(
        transforms.RandomApply(
            [transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0))], p=0.6
        )
    )
    # horizontal flip
    aug_list.append(transforms.RandomHorizontalFlip(p=0.5))
    # color jitter
    aug_list.append(
        transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.1, 0.05)], p=0.6)
    )
    # JPEG
    aug_list.append(RandomJPEG(p=0.3, q_range=(60, 95)))
    # RandAugment (if torchvision supports it) - wrapped
    try:
        from torchvision.transforms import RandAugment

        aug_list.append(transforms.RandomApply([RandAugment()], p=0.25))
    except Exception:
        pass
    # Gaussian blur sometimes
    aug_list.append(
        transforms.RandomApply(
            [
                lambda img: img.filter(
                    ImageFilter.GaussianBlur(radius=random.uniform(0.1, 1.8))
                )
            ],
            p=0.25,
        )
    )
    # Add gaussian pixel noise sometimes
    aug_list.append(RandomGaussianNoise(p=0.25, std=0.02))
    # brightness jitter more finely (RandomApply)
    aug_list.append(
        transforms.RandomApply([transforms.ColorJitter(brightness=(0.8, 1.2))], p=0.5)
    )

    # final: ensure image is resized to IMAGE_SIZE (if not already)
    final = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), *aug_list])
    return final


IMG_AUG = make_train_image_augmentations(IMAGE_SIZE)
TEST_AUG = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ]
)

def transform_img(image, target_size=512):
    tform = transforms.Compose(
        [
            transforms.Resize(target_size),
            transforms.CenterCrop(target_size),
            transforms.ToTensor(),
            transforms.ConvertImageDtype(torch.float32),
        ]
    )
    image = tform(image)
    return 2.0 * image - 1.0


# ---------------- Data loader that does augment -> pipe -> forward_diffusion -> FFT ----------------
class WatermarkOnTheFlyDataset(Dataset):
    """
    Loads image files and labels, applies augmentations, then runs:
      tsr_img -> pipe.get_image_latents(sample=False) -> pipe.forward_diffusion(...) -> FFT
    Returns: (fft_channels_tensor (float32, shape (2*C, H, W)), label)
    """

    def __init__(
        self,
        file_paths,
        labels,
        pipe,
        text_embeddings,
        num_inference_steps,
        guidance_scale=1.0,
        device="cpu",
        image_aug=IMG_AUG,
    ):
        assert len(file_paths) == len(labels)
        self.file_paths = file_paths
        self.labels = labels
        self.pipe = pipe
        self.text_embeddings = text_embeddings
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.device = device
        self.image_aug = image_aug

    def __len__(self):
        return len(self.file_paths)

    def _load_pil(self, fp):
        # accept PIL.Image, numpy array, or path
        if isinstance(fp, Image.Image):
            return fp.convert("RGB")
        if isinstance(fp, torch.Tensor):
            # convert tensor (C,H,W) to PIL
            arr = (fp.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            return Image.fromarray(arr)
        p = Path(fp)
        img = Image.open(p).convert("RGB")
        return img

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label = int(self.labels[idx])

        pil_img = self._load_pil(path)

        # --- AUGMENT IMAGE FIRST ---
        img_aug = self.image_aug(pil_img)

        # convert to tensor and to device/dtype for pipe
        tsr_img = transform_img(img_aug).unsqueeze(
            0
        )  # (1,C,H,W)
        # move to correct dtype & device for the unet/vae as the user did earlier:
        target_dtype = next(self.pipe.unet.parameters()).dtype
        tsr_img = tsr_img.to(dtype=target_dtype, device=self.device)

        # --- encode to image latents ---
        with torch.no_grad():
            image_latents = self.pipe.get_image_latents(
                tsr_img, sample=False
            )  # user's helper expects (C,H,W) or (B,C,H,W)
            # ensure batch dim
            if image_latents.ndim == 3:
                image_latents = image_latents.unsqueeze(0)

            
            # --- forward/inversion -> x_T (depending on forward_diffusion implementation)
            reversed_latents = self.pipe.forward_diffusion(
                latents=image_latents,
                text_embeddings=self.text_embeddings,
                guidance_scale=1,
                num_inference_steps=self.num_inference_steps,
            )  # expect tensor shape (B,C,H,W)


            # Keep complex-safe: cast to float32 after splitting real/imag
            # Compute FFT (complex)
            vis_latent_fft = torch.fft.fftshift(
                torch.fft.fft2(reversed_latents), dim=(-1, -2)
            )  # (B,C,H,W) complex
            # we assume batch==1
            fft_b = vis_latent_fft[0]
            # convert to float channels (real, imag) as float32
            real = fft_b.real.to(dtype=torch.float32)
            imag = fft_b.imag.to(dtype=torch.float32)
            fft_ch = torch.cat([real, imag], dim=0)  # (2*C, H, W)

        return fft_ch, torch.tensor(label, dtype=torch.long)


# ---------------- helper to collect files and labels ----------------
def discover_dataset_files(data_dir: str):
    """
    Discover files in data_dir. Support:
      - subfolders 'watermarked' and 'clean' (or any two subfolders)
      - .pt files with keys
    Returns lists: file_paths, labels
    """
    p = Path(data_dir)
    if not p.exists():
        raise RuntimeError(f"{data_dir} not found")

    # case: two subfolders inside (binary classes)
    subdirs = [d for d in p.iterdir() if d.is_dir()]
    if len(subdirs) >= 2:
        # choose the first two directories as classes
        classes = sorted(subdirs)[:2]
        file_paths = []
        labels = []
        for label, cdir in enumerate(classes):
            exts = list(cdir.glob("*"))
            for f in exts:
                if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".pt", ".pth"]:
                    file_paths.append(str(f))
                    labels.append(label)
        return file_paths, labels

    # case: many .pt files with 'fft' or 'image' and 'label'
    pts = list(p.glob("*.pt"))
    if len(pts) > 0:
        file_paths = []
        labels = []
        for f in pts:
            try:
                d = torch.load(f)
                if isinstance(d, dict) and "label" in d:
                    file_paths.append(str(f))
                    labels.append(int(d["label"]))
            except Exception:
                continue
        if len(file_paths) > 0:
            return file_paths, labels

    # fallback: collect images in folder and try to infer labels by filename (contains 'water' or 'wm')
    imgs = [
        str(f)
        for f in p.glob("*")
        if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp"]
    ]
    if len(imgs) > 0:
        file_paths = []
        labels = []
        for f in imgs:
            fname = os.path.basename(f).lower()
            lbl = 0
            if "water" in fname or "wm" in fname or "marked" in fname or "1_" in fname:
                lbl = 1
            file_paths.append(f)
            labels.append(lbl)
        return file_paths, labels

    raise RuntimeError(
        "Unable to discover dataset files. Please structure dataset as subfolders or .pt files with label key."
    )


# ---------------- small model helper ----------------
def make_model(in_channels):
    model = models.resnet18(pretrained=False)
    # adapt first conv
    model.conv1 = nn.Conv2d(
        in_channels,
        model.conv1.out_channels,
        kernel_size=model.conv1.kernel_size,
        stride=model.conv1.stride,
        padding=model.conv1.padding,
        bias=(model.conv1.bias is not None),
    )
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


# ---------------- training / eval loops ----------------
def train_epoch(model, loader, opt, crit):
    model.train()
    running_loss = 0.0
    for X, y in tqdm(loader, desc="train", leave=False):
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        opt.zero_grad()
        out = model(X)
        loss = crit(out, y)
        loss.backward()
        opt.step()
        running_loss += loss.item() * X.size(0)
    return running_loss / len(loader.dataset)

def eval_model(model, loader):
    model.eval()
    trues, preds, probs = [], [], []
    running_loss = 0.0
    with torch.no_grad():
        for X, y in tqdm(loader, desc="eval", leave=False):
            X = X.to(DEVICE)
            y = y.to(DEVICE)
            out = model(X)
            loss = crit(out, y)
            p = torch.softmax(out, dim=1)[:, 1].detach().cpu().numpy()
            pred = (p > 0.5).astype(int).tolist()
            probs.extend(p.tolist())
            preds.extend(pred)
            trues.extend(y.cpu().numpy().tolist())
            running_loss += loss.item() * X.size(0)

    acc = accuracy_score(trues, preds)
    try:
        auc = roc_auc_score(trues, probs)
    except Exception:
        auc = float("nan")
    return acc, auc, (running_loss / len(loader.dataset))

c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mike8\anaconda3\envs\pytorch2\Lib\site-packages\diffusers\pipelines\pipeline_loading_utils.py:333: FutureWarning: You are loading the variant fp16 from stabilityai/stable-diffusion-2-1-base via `revision='fp16'`. This behavior is deprecated and will be removed in diffusers v1. One should use `variant='fp16'` instead. However, it appears that stabilityai/stable-diffusion-2-1-base currently does not have the required variant filenames in the 'main' branch. 
 The Diffusers team and community would be very grateful if you could open an issue: https://github.com/huggingface/diffusers/issues/new with the title 'stabilityai/stable-diffusion-2-1-base is missing fp16 files' so that the correct variant file can be added.
  warnin

In [2]:
file_paths, labels = discover_dataset_files(DATA_DIR)
# shuffle & split
combined = list(zip(file_paths, labels))
random.shuffle(combined)
file_paths, labels = zip(*combined)
n_val = int(len(file_paths) * VALIDATION_SPLIT)
val_paths = file_paths[:n_val]
val_labels = labels[:n_val]
train_paths = file_paths[n_val:]
train_labels = labels[n_val:]

# create datasets (num_workers=0 recommended)
train_ds = WatermarkOnTheFlyDataset(
    train_paths,
    train_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
)
val_ds = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
)

# get sample to know channels
sample_fft, _ = train_ds[0]
in_ch = sample_fft.shape[0]

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

model = make_model(in_ch).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, opt, crit)
    print(f"Epoch {epoch} train_loss {train_loss:.4f}")
    acc, auc = eval_model(model, val_loader)
    print(f"Val Acc: {acc:.4f} AUROC: {auc:.4f}")

torch.save(model.state_dict(), "fft_detector_resnet18_on_the_fly.pth")
print("Saved model to fft_detector_resnet18_on_the_fly.pth")


# Training time: 882m45s

Epoch 1 train_loss 0.7507


Val Acc: 0.5000 AUROC: 0.5791


Epoch 2 train_loss 0.7195


Val Acc: 0.6067 AUROC: 0.6044


Epoch 3 train_loss 0.7016


...

Val Acc: 0.7333 AUROC: 0.8069


Epoch 49 train_loss 0.4681


Val Acc: 0.7267 AUROC: 0.8058


Epoch 50 train_loss 0.4740


Val Acc: 0.7333 AUROC: 0.8060
Saved model to fft_detector_resnet18_on_the_fly.pth


In [3]:
# Do 5 time evaluation and get average metrics.

In [4]:
# Measure the evaluation without augmentation.